In [3]:
import json
from pathlib import Path
from collections import defaultdict, Counter

from datasets import load_dataset

/Users/matthewho/miniconda3/envs/collabmem/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def read_json(file_path):
    with open(file_path, "r") as f:
        return json.load(f)

In [ ]:
# lcb_exec2 = load_dataset("livecodebench/execution-v2", split="test")

ValueError: Feature type 'List' not found. Available feature types: ['Value', 'ClassLabel', 'Translation', 'TranslationVariableLanguages', 'LargeList', 'Sequence', 'Array2D', 'Array3D', 'Array4D', 'Array5D', 'Audio', 'Image', 'Video', 'Pdf']

In [ ]:
# get distribution (Counter) of difficulties (easy, medium, hard)
difficulty_dist = {}
for example in lcb_exec2:
    difficulty = example["difficulty"]
    if difficulty not in difficulty_dist:
        difficulty_dist[difficulty] = 0
    difficulty_dist[difficulty] += 1
print("Difficulty distribution:", difficulty_dist)

Difficulty distribution: {'easy': 216, 'medium': 254, 'hard': 9}


In [ ]:
lcb_exec2

Dataset({
    features: ['question_id', 'id', 'function_name', 'code', 'input', 'output', 'numsteps', 'problem_id', 'contest_id', 'contest_date', 'difficulty'],
    num_rows: 479
})

In [6]:
# make a dict that maps "question_id" -> original record
lcb_exec2_dict = {}
for example in lcb_exec2:
    question_id = example["question_id"]
    lcb_exec2_dict[question_id] = example

NameError: name 'lcb_exec2' is not defined

In [24]:
original_lic_dataset_path = Path.cwd().parent / "data/sharded_instructions_600.json"
print("Original LIC dataset path:", original_lic_dataset_path)
lic_dataset = read_json(original_lic_dataset_path)

Original LIC dataset path: /Users/matthewho/Documents/research/ctx_editor/data/sharded_instructions_600.json


Question ID 2755 not found in LCB Exec2 dataset.
Question ID 2791 not found in LCB Exec2 dataset.
Question ID 2802 not found in LCB Exec2 dataset.
Question ID 2812 not found in LCB Exec2 dataset.
Question ID 2844 not found in LCB Exec2 dataset.
Question ID 2872 not found in LCB Exec2 dataset.
Question ID 2873 not found in LCB Exec2 dataset.
Question ID 2877 not found in LCB Exec2 dataset.
Question ID 2882 not found in LCB Exec2 dataset.
Question ID 2893 not found in LCB Exec2 dataset.
Question ID 2998 not found in LCB Exec2 dataset.
Number of HumanEval examples: 45
LCB difficulty distribution in LIC dataset: Counter({'medium': 26, 'easy': 18})


In [14]:
from datasets import load_dataset
lcb_codegen = load_dataset("livecodebench/code_generation_lite", version_tag="release_v6")

Generating test split: 1055 examples [01:32, 11.35 examples/s]


In [27]:
# want to save release_v6 to a json file so I can still use this with hf datasets>=4.0.0
# which doesn't support trust_remote_code or whatever (originally had to downgrade to 3.x to load this)
general_datasets_path = Path("/Users/matthewho/Documents/research/datasets")
lcb_codegen["test"].to_json(general_datasets_path / "lcb_codegen_v6.json")

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Creating json from Arrow format: 100%|██████████| 2/2 [03:20<00:00, 100.09s/ba]


4555700601

In [15]:
lcb_codegen_test = lcb_codegen["test"]

In [18]:
lcb_codegen_test

Dataset({
    features: ['question_title', 'question_content', 'platform', 'question_id', 'contest_id', 'contest_date', 'starter_code', 'difficulty', 'public_test_cases', 'private_test_cases', 'metadata'],
    num_rows: 1055
})

In [19]:
for i in range(10):
    print(lcb_codegen_test[i]["question_id"])

1873_A
1873_B
1873_D
1883_B
1883_C
1899_A
1899_B
1899_C
1899_D
2727


In [16]:
# get difficulty distribution in lcb_codegen_test
lcb_codegen_difficulty_dist = Counter()
for example in lcb_codegen_test:
    difficulty = example["difficulty"]
    lcb_codegen_difficulty_dist[difficulty] += 1

In [17]:
print("LCB CodeGen difficulty distribution in test set:", lcb_codegen_difficulty_dist)

LCB CodeGen difficulty distribution in test set: Counter({'medium': 383, 'hard': 350, 'easy': 322})


In [20]:
# create question_id -> difficulty dict for lcb_codegen_test
lcb_codegen_difficulty_dict = {}
for example in lcb_codegen_test:
    question_id = example["question_id"]
    difficulty = example["difficulty"]
    lcb_codegen_difficulty_dict[question_id] = difficulty

In [23]:
# count how how many question_ids fail to convert to int
failed_to_convert = 0
max_count = 30
for question_id in lcb_codegen_difficulty_dict.keys():
    try:
        int(question_id)
    except ValueError:
        # print(f"Question ID {question_id} failed to convert to int.")
        failed_to_convert += 1
        # try again but split on "_" and take the first part
        try:
            int(question_id.split("_")[0])
        except ValueError:
            print(f"Question ID {question_id} failed to convert to int even after splitting.")
            if failed_to_convert >= max_count:
                break
print("Number of question_ids that failed to convert to int:", failed_to_convert)

Question ID abc301_a failed to convert to int even after splitting.
Question ID abc301_b failed to convert to int even after splitting.
Question ID abc301_c failed to convert to int even after splitting.
Question ID abc301_d failed to convert to int even after splitting.
Question ID abc301_e failed to convert to int even after splitting.
Question ID abc301_f failed to convert to int even after splitting.
Question ID abc302_a failed to convert to int even after splitting.
Question ID abc302_b failed to convert to int even after splitting.
Question ID abc302_c failed to convert to int even after splitting.
Question ID abc302_d failed to convert to int even after splitting.
Question ID abc302_e failed to convert to int even after splitting.
Question ID abc302_f failed to convert to int even after splitting.
Question ID abc303_a failed to convert to int even after splitting.
Question ID abc303_b failed to convert to int even after splitting.
Question ID abc303_c failed to convert to int ev

In [25]:
humaneval_count = 0
lcb_difficulty_dist = Counter()

for example in lic_dataset:
    if example["task"] != "code":
        continue
    # either "sharded-HumanEval/{id}"
    # or "sharded-livecodebench/{id}"
    # print(example["task_id"])
    if example["task_id"].startswith("sharded-HumanEval/"):
        humaneval_count += 1
    elif example["task_id"].startswith("sharded-livecodebench/"):
        # extract the question_id from the task_id
        question_id = example["task_id"].split("/")[1]
        if question_id in lcb_codegen_difficulty_dict:
            difficulty = lcb_codegen_difficulty_dict[question_id]
            lcb_difficulty_dist[difficulty] += 1
        else:
            print(f"Question ID {question_id} not found in LCB Exec2 dataset.")
    else:
        print(f"Unknown task_id format: {example['task_id']}")

print("Number of HumanEval examples:", humaneval_count)
print("LCB difficulty distribution in LIC dataset:", lcb_difficulty_dist)

Number of HumanEval examples: 45
LCB difficulty distribution in LIC dataset: Counter({'medium': 33, 'easy': 22})
